In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import *
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

In [2]:
import os
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.datasets import load_iris

In [3]:
spark = SparkSession.builder.appName("Clustering") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 11:09:46 WARN Utils: Your hostname, Keshabs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.13.30.17 instead (on interface en0)
26/06/26 11:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 11:09:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 11:09:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57249)
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 318, in

In [4]:
iris_sk = load_iris()

feature_names = ["sepal_length" , "sepal_width" , "petal_length", "petal_width"]

target_names = iris_sk.target_names

In [5]:
iris_pd = pd.DataFrame(iris_sk.data, columns= feature_names)

In [6]:
iris_pd["true_label"] = iris_sk.target
iris_pd["species"] = iris_pd["true_label"].map({i: str(name) for i, name in enumerate(target_names)})

In [7]:
iris_pd["species"] = iris_pd["species"].astype(str)

In [8]:
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, StringType

schema = StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width", DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width", DoubleType(), True),
    StructField("true_label", LongType(), True),
    StructField("species", StringType(), True)
]
)

In [9]:
iris_spark = spark.createDataFrame(iris_pd,  schema = schema)

In [10]:
iris_spark.show(5)

[Stage 0:>                                                          (0 + 1) / 1]

+------------+-----------+------------+-----------+----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|true_label|species|
+------------+-----------+------------+-----------+----------+-------+
|         5.1|        3.5|         1.4|        0.2|         0| setosa|
|         4.9|        3.0|         1.4|        0.2|         0| setosa|
|         4.7|        3.2|         1.3|        0.2|         0| setosa|
|         4.6|        3.1|         1.5|        0.2|         0| setosa|
|         5.0|        3.6|         1.4|        0.2|         0| setosa|
+------------+-----------+------------+-----------+----------+-------+
only showing top 5 rows


In [11]:
assembler = VectorAssembler(inputCols=feature_names, outputCol="features")
df_assembled = assembler.transform(iris_spark)

In [12]:
scaler = StandardScaler(inputCol= "features" , outputCol= "scaled_features", 
                       withMean= True, withStd= True)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

In [13]:
df_scaled.show()

+------------+-----------+------------+-----------+----------+-------+-----------------+--------------------+
|sepal_length|sepal_width|petal_length|petal_width|true_label|species|         features|     scaled_features|
+------------+-----------+------------+-----------+----------+-------+-----------------+--------------------+
|         5.1|        3.5|         1.4|        0.2|         0| setosa|[5.1,3.5,1.4,0.2]|[-0.8976738791967...|
|         4.9|        3.0|         1.4|        0.2|         0| setosa|[4.9,3.0,1.4,0.2]|[-1.1392004834649...|
|         4.7|        3.2|         1.3|        0.2|         0| setosa|[4.7,3.2,1.3,0.2]|[-1.3807270877331...|
|         4.6|        3.1|         1.5|        0.2|         0| setosa|[4.6,3.1,1.5,0.2]|[-1.5014903898672...|
|         5.0|        3.6|         1.4|        0.2|         0| setosa|[5.0,3.6,1.4,0.2]|[-1.0184371813308...|
|         5.4|        3.9|         1.7|        0.4|         0| setosa|[5.4,3.9,1.7,0.4]|[-0.5353839727944...|
|         

In [14]:
# Cluster Assumption 3
 
kmeans_model = KMeans(k=3, featuresCol="scaled_features",
                       predictionCol="km_cluster", seed=42, maxIter=200)
kmeans_model = kmeans_model.fit(df_scaled)

df_km = kmeans_model.transform(df_scaled)

26/06/26 11:12:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [15]:
df_km.groupBy("km_cluster").count().orderBy("km_cluster").show()

+----------+-----+
|km_cluster|count|
+----------+-----+
|         0|   50|
|         1|   56|
|         2|   44|
+----------+-----+



In [16]:
from pyspark.ml.evaluation import *

evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol= "features",
    metricName="silhouette",
    distanceMeasure= "squaredEuclidean"
)

km_sil = evaluator.evaluate(df_km.withColumnRenamed("km_cluster", "prediction"))
print(km_sil)

0.6567724235382677


In [17]:
### Visualisation

km_pd = df_km.select(feature_names + ["species", "km_cluster"]).toPandas()

In [18]:
km_pd

,sepal_length,sepal_width,petal_length,petal_width,species,km_cluster
0,5.1,3.5,1.4,0.2,setosa,0
1,4.9,3.0,1.4,0.2,setosa,0
2,4.7,3.2,1.3,0.2,setosa,0
3,4.6,3.1,1.5,0.2,setosa,0
4,5.0,3.6,1.4,0.2,setosa,0
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica,2
146,6.3,2.5,5.0,1.9,virginica,1
147,6.5,3.0,5.2,2.0,virginica,2
148,6.2,3.4,5.4,2.3,virginica,2


In [19]:
export_df = iris_pd.copy()

In [20]:
export_df["km_cluster"] = km_pd["km_cluster"].values

In [21]:
km_map = {0: "KM_Cluster_0", 1: "KM_Cluster_1", 2: "KM_Cluster_2"}

In [22]:
export_df["km_cluster_name"] = export_df["km_cluster"].map(km_map)

In [23]:
output_path = "iris_clustering_results.csv"
export_df.to_csv(output_path, index = False)